In [1]:
import os
import re
import pandas as pd
import pytesseract
from pdf2image import convert_from_path

# --- CONFIGURATION ---
folder_path = "/Users/sushmitarajan/Downloads/all_waivers"

def clean_statute(s):
    # Remove OCR noise
    s = re.sub(r'[\s|_|I|l|\||\[|\]|~|=]+', '', s)
    # Standardize Georgia Code format
    if s.startswith('20-2-'):
        s = '§ ' + s
    elif s.startswith('§') or s.startswith('8'):
        content = re.sub(r'^[§8]+', '', s)
        s = '§ ' + content
    s = re.sub(r'[.\-]+$', '', s)
    return s

def find_statutes_in_text(text, current_date, page_num, county_name):
    # Regex for Georgia Statutes and Board Rules
    pattern = r'(?:§|8)?\s*(?:20|160)[\d\s\-\.\(\)a-z]*[\d\)]'
    matches = re.findall(pattern, text)
    
    rows = []
    for m in matches:
        clean = clean_statute(m)
        if clean.count('-') >= 2:
            rows.append({
                "County": county_name,
                "Date": current_date,
                "Statute": clean,
                "Page": page_num
            })
    return rows

# --- BATCH EXECUTION ---
all_final_data = []
pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')]

print(f"🚀 Starting Master Audit of {len(pdf_files)} counties...")

for index, filename in enumerate(pdf_files, start=1):
    county_clean = filename.replace('.pdf', '').replace(' Waiver', '').strip()
    file_path = os.path.join(folder_path, filename)
    
    print(f"[{index}/{len(pdf_files)}] Processing {county_clean}...", end="\r")
    
    try:
        pages = convert_from_path(file_path, dpi=300)
        latest_date = "Date Not Found"
        county_raw_collection = []
        
        for i, page in enumerate(pages, start=1):
            # Scan both orientations
            text_p = pytesseract.image_to_string(page, config='--psm 6')
            text_l = pytesseract.image_to_string(page.rotate(270, expand=True), config='--psm 6')
            full_text = text_p + " " + text_l
            
            # 1. ALWAYS update date from early pages (1-5)
            date_match = re.search(r'([A-Z][a-z]+ \d{1,2}, \d{4})', full_text)
            if date_match:
                latest_date = date_match.group(1)

            # 2. START collecting only on Page 12
            if i >= 12:
                found = find_statutes_in_text(full_text, latest_date, i, county_clean)
                county_raw_collection.extend(found)
        
        # 3. SLICE: Remove first 8 boilerplate statutes for this specific county
        if len(county_raw_collection) > 8:
            county_df = pd.DataFrame(county_raw_collection).drop_duplicates()
            # This is the "Slice" that removes the first 8 rows
            relevant_rows = county_df.iloc[8:] 
            all_final_data.extend(relevant_rows.to_dict('records'))
        else:
            # If a file is weirdly short, we keep what we found
            all_final_data.extend(county_raw_collection)

    except Exception as e:
        print(f"\n❌ Error in {filename}: {e}")

# --- FINAL SAVE ---
if all_final_data:
    master_df = pd.DataFrame(all_final_data)
    output_name = "Master_Waiver_Audit_All_Counties.csv"
    master_df.to_csv(output_name, index=False)
    
    print(f"\n\n✅ DONE! Total relevant statutes extracted: {len(master_df)}")
    print(f"📂 Master file saved as: {output_name}")
    
    # Quick summary of counts
    print("\nSummary of extracted waivers per county:")
    print(master_df.groupby('County')['Statute'].count())
else:
    print("\n\n⚠️ No data was extracted. Check folder path or PDF content.")

🚀 Starting Master Audit of 115 counties...


KeyboardInterrupt: 